<a href="https://colab.research.google.com/github/hanokjoshua144/DEEP-LEARNING-6-SEM/blob/main/W_13.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Implement Transformer model


In [ ]:
import torch
import torch.nn as nn
import math

class SelfAttention(nn.Module):
    def __init__(self, embed_size):
        super().__init__()
        self.embed_size = embed_size
        self.values = nn.Linear(embed_size, embed_size)
        self.keys = nn.Linear(embed_size, embed_size)
        self.queries = nn.Linear(embed_size, embed_size)

    def forward(self, x):
        V = self.values(x)
        K = self.keys(x)
        Q = self.queries(x)

        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.embed_size)
        attention = torch.softmax(scores, dim=-1)

        out = torch.matmul(attention, V)
        return out

class TransformerBlock(nn.Module):
    def __init__(self, embed_size):
        super().__init__()
        self.attention = SelfAttention(embed_size)
        self.norm1 = nn.LayerNorm(embed_size)
        self.ff = nn.Sequential(
            nn.Linear(embed_size, 4*embed_size),
            nn.ReLU(),
            nn.Linear(4*embed_size, embed_size)
        )
        self.norm2 = nn.LayerNorm(embed_size)

    def forward(self, x):
        attn = self.attention(x)
        x = self.norm1(attn + x)
        ff = self.ff(x)
        return self.norm2(ff + x)

x = torch.rand(2, 5, 16)  # batch=2, seq_len=5, embed=16
model = TransformerBlock(16)
out = model(x)
print(out.shape)

BERT Model (Multi-task)

(a) Next Word Prediction

In [ ]:
from transformers import pipeline

predictor = pipeline("fill-mask", model="bert-base-uncased")

result = predictor("I love learning [MASK]")
print(result)

(b) Missing Word Prediction

In [ ]:
text = "The capital of India is [MASK]."
print(predictor(text))

(c) Review Classification

In [ ]:
from transformers import pipeline

classifier = pipeline("sentiment-analysis")

result = classifier("This product is amazing!")
print(result)

3. Vision Transformer (ViT)

In [ ]:
import torch
import torch.nn as nn

class ViT(nn.Module):
    def __init__(self, img_size=28, patch_size=7, embed_dim=64):
        super().__init__()
        self.patch_size = patch_size
        self.num_patches = (img_size // patch_size) ** 2

        self.projection = nn.Linear(patch_size*patch_size, embed_dim)
        self.transformer = nn.TransformerEncoder(
            nn.TransformerEncoderLayer(d_model=embed_dim, nhead=4),
            num_layers=2
        )
        self.classifier = nn.Linear(embed_dim, 10)

    def forward(self, x):
        patches = x.unfold(2, self.patch_size, self.patch_size)\
                   .unfold(3, self.patch_size, self.patch_size)
        patches = patches.contiguous().view(x.size(0), -1, self.patch_size*self.patch_size)

        x = self.projection(patches)
        x = self.transformer(x)
        x = x.mean(dim=1)

        return self.classifier(x)

img = torch.rand(1, 1, 28, 28)
model = ViT()
out = model(img)
print(out)

Implement GAN model on MNIST.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt

# Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Hyperparameters
lr = 0.0002
batch_size = 64
epochs = 5
z_dim = 100

# MNIST Dataset
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

dataset = datasets.MNIST(root="./data", train=True, transform=transform, download=True)
loader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

# Generator
class Generator(nn.Module):
    def __init__(self, z_dim):
        super().__init__()
        self.model = nn.Sequential(
            nn.Linear(z_dim, 256),
            nn.ReLU(),
            nn.Linear(256, 512),
            nn.ReLU(),
            nn.Linear(512, 784),
            nn.Tanh()
        )

    def forward(self, x):
        return self.model(x)

# Discriminator
class Discriminator(nn.Module):
    def __init__(self):
        super().__init__()
        self.model = nn.Sequential(
            nn.Linear(784, 512),
            nn.LeakyReLU(0.2),
            nn.Linear(512, 256),
            nn.LeakyReLU(0.2),
            nn.Linear(256, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.model(x)

# Initialize models
G = Generator(z_dim).to(device)
D = Discriminator().to(device)

# Loss and Optimizers
criterion = nn.BCELoss()
opt_G = optim.Adam(G.parameters(), lr=lr)
opt_D = optim.Adam(D.parameters(), lr=lr)

# Training Loop
for epoch in range(epochs):
    for batch_idx, (real, _) in enumerate(loader):
        real = real.view(-1, 784).to(device)
        batch_size = real.size(0)

        # Labels
        real_labels = torch.ones(batch_size, 1).to(device)
        fake_labels = torch.zeros(batch_size, 1).to(device)

        # -------- Train Discriminator --------
        noise = torch.randn(batch_size, z_dim).to(device)
        fake = G(noise)

        loss_real = criterion(D(real), real_labels)
        loss_fake = criterion(D(fake.detach()), fake_labels)
        loss_D = loss_real + loss_fake

        opt_D.zero_grad()
        loss_D.backward()
        opt_D.step()

        # -------- Train Generator --------
        output = D(fake)
        loss_G = criterion(output, real_labels)

        opt_G.zero_grad()
        loss_G.backward()
        opt_G.step()

    print(f"Epoch [{epoch+1}/{epochs}]  Loss_D: {loss_D:.4f}, Loss_G: {loss_G:.4f}")

# Generate Images
noise = torch.randn(16, z_dim).to(device)
fake_images = G(noise).reshape(-1, 1, 28, 28)

# Plot Results
fig, axes = plt.subplots(4, 4)
for i, ax in enumerate(axes.flatten()):
    ax.imshow(fake_images[i].detach().cpu().squeeze(), cmap="gray")
    ax.axis("off")

plt.show()